### Import Libraries

In [3]:
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait as WDW
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager

### Setup and Configure Selenium Driver

In [4]:
print("Configuration Webdriver....")
chrome_opt = Options() # Initialize the chrome webdriver
chrome_opt.add_argument("--headless")
chrome_opt.add_argument("--disable-gpu")
chrome_opt.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/131.0.6778.265 Safari/537.36")
print("configuration done!")

Configuration Webdriver....
configuration done!


In [5]:
# Setting Up webdriver: Installation and Initialization
print("Installing Chrome Webdriver")
service = Service(ChromeDriverManager().install())
print("Final Setup")
driver = webdriver.Chrome(service=service, options=chrome_opt)
print("done!!")

Installing Chrome Webdriver
Final Setup
done!!


### Making a Connection to the Webpage

In [6]:
URL = "https://www.glasses.com/gl-us/eyeglasses"

In [7]:
print(f"Visiting {URL} page")
driver.get(URL)
print("Page loaded successfully!")

# Further Instructions
try:
    print("Waiting for product tiles to load")
    WDW(driver, 10).until(EC.presence_of_element_located((By.CLASS_NAME, 'catalog-page')))
    print("Done, Proceed!")
except TimeoutError as e:
    print(f"Expected tag did not load on time: {e}")

Visiting https://www.glasses.com/gl-us/eyeglasses page
Page loaded successfully!
Waiting for product tiles to load
Done, Proceed!


In [8]:
content = driver.page_source
page = BeautifulSoup(content, "html.parser")

### Data Extraction

In [9]:
product_tiles = page.find_all('a', class_='product-tile')
print(f"Found {len(product_tiles)} products")

Found 26 products


In [12]:
products = []

for tile in product_tiles:
    product_info = tile.find('div', class_='product-info')

    # Default values
    name = "Unknown"
    brand = "Unknown"
    former_price = "Unknown"
    current_price = "Unknown"
    frame_type = "Unknown"
    image_url = "Unknown"

    if product_info:
        # product name
        name_tag = product_info.find('div', class_='product-code')
        name = name_tag.text.strip() if name_tag else "Unknown"

        # brand
        brand_tag = product_info.find('div', class_='product-brand')
        brand = brand_tag.text.strip() if brand_tag else "Unknown"

        # frame type
        frame_tag = product_info.find('div', class_='product-frame-type')
        frame_type = frame_tag.text.strip() if frame_tag else "Unknown"

        # price
        price_container = product_info.find('div', class_='product-prices-container')
        if price_container:
            # former price
            former_price_tag = price_container.find('div', class_='product-list-price')
            former_price = former_price_tag.text.strip() if former_price_tag else "Unknown"

            # current price
            current_price_tag = price_container.find('div', class_='product-offer-price')
            current_price = current_price_tag.text.strip() if current_price_tag else "Unknown"

    # product image
    image_tag = tile.find('img')
    image_url = image_tag.get('src').strip() if image_tag and image_tag.get('src') else "Unknown"

    # product link
    base_url = "https://www.glasses.com"
    product_link = base_url + tile.get('href', '')

    # discount if available
    discount_tag = tile.find('div', class_='product-badge discount-badge')
    discount = discount_tag.text.strip() if discount_tag else None

    # final product record
    data = {
        'Vendor': 'Glasses.com',
        'Brand': brand,
        'Product_Name': name,
        'Former_Price': former_price,
        'Current_Price': current_price,
        'Frame_Type': frame_type,
        'Product_Image': image_url,
        'Discount': discount,
        'Product_Link': product_link
    }

    print(data)
    products.append(data)


{'Vendor': 'Glasses.com', 'Brand': 'Dolce & Gabbana', 'Product_Name': 'DG3404', 'Former_Price': '$ 422.00', 'Current_Price': '$ 295.40', 'Frame_Type': 'Unknown', 'Product_Image': 'https://assets2.glasses.com/cdn-record-files-pi/62b01bfb-a9f4-4b6a-9a79-b17d0160f452/1f6b66b6-55da-4ba5-83be-b17d016ae023/0DG3404__501__P21__noshad__qt.png?imwidth=400', 'Discount': None, 'Product_Link': 'https://www.glasses.comhttps://www.glasses.com/gl-us/dolce-and-gabbana/8056262242971'}
{'Vendor': 'Glasses.com', 'Brand': 'Michael Kors', 'Product_Name': 'MK3068 Portland', 'Former_Price': '$ 175.00', 'Current_Price': '$ 87.50', 'Frame_Type': 'Unknown', 'Product_Image': 'https://assets2.glasses.com/cdn-record-files-pi/54193723-fc16-4084-94cc-af1c0101b07a/d6c10b94-0291-4b88-ab40-af1d0001be79/0MK3068__1108__P21__noshad__qt.png?imwidth=400', 'Discount': None, 'Product_Link': 'https://www.glasses.comhttps://www.glasses.com/gl-us/michael-kors/725125395281'}
{'Vendor': 'Glasses.com', 'Brand': 'Polo Ralph Lauren', 

In [13]:
# Step 3 - Data Storage: store the extracted data in CSV and JSON formats
import csv
import json
# Save to CSV file
column_name = products[0].keys() # get the column names
with open('glassesdotcom.csv', mode='w', newline='', encoding='utf-8') as csv_file: # open up the file with context manager
    dict_writer = csv.DictWriter(csv_file, fieldnames=column_name)
    dict_writer.writeheader()
    dict_writer.writerows(products)
print(f"Saved {len(products)} records to CSV in the extracted data folder.")

# Save to JSON file
with open("glassesdotcom.json", mode='w') as json_file:
    json.dump(products, json_file, indent=4)
print(f"Saved {len(products)} records to JSON in the extracted data folder.")

# close the browser
driver.quit()
print("End of Web Extraction")

Saved 26 records to CSV in the extracted data folder.
Saved 26 records to JSON in the extracted data folder.
End of Web Extraction
